# EvidenceMem: one-notebook Colab experiment suite

**Purpose.** Test whether a bounded memory of reliability-weighted, displayable
CLIP medoids can support classification, visual evidence retrieval, unknown-input
rejection, and class insertion without updating the encoder.

This notebook is self-contained so it can run even while the GitHub repository is
private. It does not assume the result is positive. It saves tables, per-example
predictions, figures, configuration, environment details, and an experiment journal.

[Open the committed notebook in Colab](https://colab.research.google.com/github/Prathmesh333/evidencemem/blob/main/notebooks/EvidenceMem_Colab_T4.ipynb)

## Before running

1. In Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells in order. `RUN_MODE="validation"` is the bounded end-to-end check.
3. After the validation run succeeds, change `RUN_MODE` to `"paper"` for the full
   CIFAR-10 experiment with three seeds. Expect the paper run to take hours, mostly
   because CLIP encoding and repeated CPU clustering are real computations.
4. Hyperparameters are chosen on the validation split only. Test labels are never
   used for selection. CIFAR-100 and SVHN are used only as OOD test sets here.

A Colab T4 has 16 GB VRAM. The embedding batch size below is deliberately conservative.
If Colab gives a different CUDA GPU, the notebook will report it and continue.

In [ ]:
# Install only missing experiment dependencies. Torch and torchvision are supplied by Colab.
import importlib.util
import subprocess
import sys

REQUIRED = {
    "open_clip": "open_clip_torch>=2.30,<4",
    "faiss": "faiss-cpu>=1.8,<2",
    "sklearn": "scikit-learn>=1.4,<2",
    "pandas": "pandas>=2.0,<3",
    "seaborn": "seaborn>=0.13,<1",
    "scipy": "scipy>=1.11,<2",
    "tqdm": "tqdm>=4.66,<5",
    "psutil": "psutil>=5.9,<8",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready.")

In [ ]:
# Configuration: this is the only cell that normally needs editing.
import os
from dataclasses import asdict, dataclass
from pathlib import Path

RUN_MODE = "validation"  # change to "paper" only after the validation run passes
USE_GOOGLE_DRIVE = False  # True persists caches across Colab sessions (prompts for authorization)
RUN_RESNET18 = RUN_MODE == "paper"  # compute-matched supervised reference, not part of EvidenceMem
RUN_CIFAR100_CLASSIFICATION = RUN_MODE == "paper"  # secondary scaling experiment
RUN_NOISE_ROBUSTNESS = True
OPENCLIP_MODEL = "ViT-B-32"
OPENCLIP_WEIGHTS = "openai"


@dataclass(frozen=True)
class RunConfig:
    mode: str
    seeds: tuple
    train_size: int
    val_size: int
    test_size: int
    ood_size: int
    budgets: tuple
    default_budget: int
    topk_grid: tuple
    alpha_grid: tuple
    temperatures: tuple
    insertion_shots: tuple
    embedding_batch_size: int
    num_workers: int
    resnet_epochs: int


if RUN_MODE == "validation":
    CFG = RunConfig(
        mode=RUN_MODE,
        seeds=(7,),
        train_size=12_000,
        val_size=2_000,
        test_size=2_000,
        ood_size=2_000,
        budgets=(1, 5, 10),
        default_budget=10,
        topk_grid=(5, 10, 20),
        alpha_grid=(0.0, 0.25, 0.5, 0.75, 1.0),
        temperatures=(0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.50, 1.0, 2.0),
        insertion_shots=(1, 5, 10),
        embedding_batch_size=128,
        num_workers=2,
        resnet_epochs=2,
    )
elif RUN_MODE == "paper":
    CFG = RunConfig(
        mode=RUN_MODE,
        seeds=(7, 17, 29),
        train_size=45_000,
        val_size=5_000,
        test_size=10_000,
        ood_size=10_000,
        budgets=(1, 2, 5, 10, 20, 50, 100, 250),
        default_budget=20,
        topk_grid=(5, 10, 20, 50, 100),
        alpha_grid=(0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0),
        temperatures=(0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.50, 1.0, 2.0),
        insertion_shots=(1, 5, 10, 25),
        embedding_batch_size=128,
        num_workers=2,
        resnet_epochs=12,
    )
else:
    raise ValueError("RUN_MODE must be 'validation' or 'paper'.")

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/EvidenceMem")
else:
    ROOT = Path("/content/EvidenceMem")

DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "cache"
RUN_DIR = ROOT / "runs" / CFG.mode
for directory in (DATA_DIR, CACHE_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(asdict(CFG))
print("Artifacts:", RUN_DIR)

In [ ]:
# Imports, determinism, hardware checks, and crash-safe result helpers.
import hashlib
import json
import math
import platform
import random
import tempfile
import time
from datetime import UTC, datetime

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import sklearn
import torch
import torch.nn.functional as F
import torchvision
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

sns.set_theme(context="notebook", style="whitegrid")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA is unavailable. In Colab select Runtime > Change runtime type > T4 GPU."
    )
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME.upper():
    print(f"Warning: requested a T4, but Colab assigned {GPU_NAME!r}; continuing on CUDA.")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)


def atomic_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def journal(event, **payload):
    record = {"time_utc": datetime.now(UTC).isoformat(), "event": event, **payload}
    with (RUN_DIR / "experiment_journal.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, default=str) + "\n")


environment = {
    "config": asdict(CFG),
    "device": str(DEVICE),
    "gpu": GPU_NAME,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "sklearn": sklearn.__version__,
    "faiss": getattr(faiss, "__version__", "unknown"),
    "cpu_count": psutil.cpu_count(),
    "ram_gb": round(psutil.virtual_memory().total / 2**30, 2),
    "openclip_model": OPENCLIP_MODEL,
    "openclip_weights": OPENCLIP_WEIGHTS,
}
atomic_json(RUN_DIR / "environment.json", environment)
journal("run_started", **environment)
environment

## 1. Frozen CLIP embeddings and persistent split indices

Every method below receives exactly the same normalized image embeddings. The official
CIFAR-10 test set remains untouched by tuning. The persisted indices make resumed runs
use the same examples. CIFAR-100 is near-OOD and SVHN is far-OOD; neither is used to
choose the classification model.

In [ ]:
import open_clip
from torchvision.datasets import CIFAR10, CIFAR100, SVHN

seed_everything(CFG.seeds[0])
model, _, clip_preprocess = open_clip.create_model_and_transforms(
    OPENCLIP_MODEL, pretrained=OPENCLIP_WEIGHTS, device=DEVICE
)
tokenizer = open_clip.get_tokenizer(OPENCLIP_MODEL)
model.eval().requires_grad_(False)

CLASS_NAMES = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)
PROMPTS = (
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a small photo of a {}.",
    "a close-up photo of a {}.",
    "a bright photo of a {}.",
    "a cropped photo of a {}.",
)


def encode_text_prototypes(class_names):
    class_vectors = []
    with torch.inference_mode():
        for name in class_names:
            tokens = tokenizer([template.format(name) for template in PROMPTS]).to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                vectors = model.encode_text(tokens)
            vectors = F.normalize(vectors.float(), dim=-1)
            class_vectors.append(F.normalize(vectors.mean(0), dim=0).cpu().numpy())
    return normalize(np.stack(class_vectors))


TEXT_PROTOTYPES = encode_text_prototypes(CLASS_NAMES)

c10_train = CIFAR10(DATA_DIR, train=True, transform=clip_preprocess, download=True)
c10_test = CIFAR10(DATA_DIR, train=False, transform=clip_preprocess, download=True)
c100_test = CIFAR100(DATA_DIR, train=False, transform=clip_preprocess, download=True)
svhn_test = SVHN(DATA_DIR, split="test", transform=clip_preprocess, download=True)
c10_train_raw = CIFAR10(DATA_DIR, train=True, transform=None, download=False)
c10_test_raw = CIFAR10(DATA_DIR, train=False, transform=None, download=False)


def choose_stratified(labels, train_size, test_size=0, seed=2026):
    labels = np.asarray(labels)
    if int(train_size) >= len(labels) and not test_size:
        return np.arange(len(labels), dtype=np.int64), None
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=int(train_size),
        test_size=int(test_size) if test_size else None,
        random_state=seed,
    )
    first, second = next(splitter.split(np.zeros(len(labels)), labels))
    return np.sort(first), np.sort(second) if test_size else None


split_file = CACHE_DIR / f"split_c10_{CFG.train_size}_{CFG.val_size}_{CFG.test_size}.npz"
if split_file.exists():
    split_data = np.load(split_file)
    train_idx, val_idx, test_idx = split_data["train"], split_data["val"], split_data["test"]
else:
    train_idx, val_idx = choose_stratified(c10_train.targets, CFG.train_size, CFG.val_size)
    test_idx, _ = choose_stratified(c10_test.targets, CFG.test_size)
    np.savez_compressed(split_file, train=train_idx, val=val_idx, test=test_idx)

c100_idx, _ = choose_stratified(c100_test.targets, min(CFG.ood_size, len(c100_test)))
svhn_idx, _ = choose_stratified(svhn_test.labels, min(CFG.ood_size, len(svhn_test)))
splits = {
    "c10_train": Subset(c10_train, train_idx.tolist()),
    "c10_val": Subset(c10_train, val_idx.tolist()),
    "c10_test": Subset(c10_test, test_idx.tolist()),
    "c100_test": Subset(c100_test, c100_idx.tolist()),
    "svhn_test": Subset(svhn_test, svhn_idx.tolist()),
}
print({name: len(ds) for name, ds in splits.items()})

In [ ]:
def cache_key(name, dataset):
    payload = f"{name}|{len(dataset)}|{OPENCLIP_MODEL}|{OPENCLIP_WEIGHTS}|224"
    return hashlib.sha256(payload.encode()).hexdigest()[:12]


def encode_dataset(name, dataset):
    path = CACHE_DIR / f"{name}_{cache_key(name, dataset)}.npz"
    if path.exists():
        cached = np.load(path)
        return normalize(cached["embeddings"]), cached["labels"].astype(np.int64)
    loader = DataLoader(
        dataset,
        batch_size=CFG.embedding_batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )
    embeddings, labels = [], []
    with torch.inference_mode():
        for images, batch_labels in tqdm(loader, desc=f"CLIP: {name}"):
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                batch_embeddings = model.encode_image(images)
            embeddings.append(F.normalize(batch_embeddings.float(), dim=-1).cpu().numpy())
            labels.append(np.asarray(batch_labels))
    x = normalize(np.concatenate(embeddings))
    y = np.concatenate(labels).astype(np.int64)
    with tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix=".npz", delete=False) as handle:
        tmp_name = handle.name
    np.savez_compressed(tmp_name, embeddings=x.astype(np.float16), labels=y)
    os.replace(tmp_name, path)
    atomic_json(
        path.with_suffix(".manifest.json"),
        {
            "dataset": name,
            "rows": len(y),
            "dimension": x.shape[1],
            "model": OPENCLIP_MODEL,
            "weights": OPENCLIP_WEIGHTS,
            "normalized": True,
            "cache_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        },
    )
    return x, y


X_train, y_train = encode_dataset("c10_train", splits["c10_train"])
X_val, y_val = encode_dataset("c10_val", splits["c10_val"])
X_test, y_test = encode_dataset("c10_test", splits["c10_test"])
X_c100, y_c100 = encode_dataset("c100_test", splits["c100_test"])
X_svhn, y_svhn = encode_dataset("svhn_test", splits["svhn_test"])
assert np.allclose(np.linalg.norm(X_train[:100], axis=1), 1.0, atol=2e-3)
journal(
    "embeddings_ready",
    train=len(y_train),
    val=len(y_val),
    test=len(y_test),
    dimension=X_train.shape[1],
)
print("Embedding shapes:", X_train.shape, X_val.shape, X_test.shape, X_c100.shape, X_svhn.shape)

## 2. EvidenceMem and equal-budget baselines

K-means centers are converted to actual training examples (medoids), so every stored
item can be displayed. Reliability combines cluster compactness, local label purity,
and CLIP text alignment. `alpha=0` is visual-only and `alpha=1` is text-only.

In [ ]:
def minmax(values):
    values = np.asarray(values, dtype=np.float32)
    lo, hi = float(values.min()), float(values.max())
    return np.ones_like(values) if hi - lo < 1e-8 else (values - lo) / (hi - lo)


def fit_memory(x, y, budget, method="evidencemem", seed=7, class_ids=None, text_prototypes=None):
    x, y = normalize(x), np.asarray(y, dtype=np.int64)
    text_prototypes = TEXT_PROTOTYPES if text_prototypes is None else normalize(text_prototypes)
    class_ids = np.unique(y) if class_ids is None else np.asarray(class_ids)
    all_index = faiss.IndexFlatIP(x.shape[1])
    all_index.add(x)
    protos, labels, sources, compactness, purity, alignment = [], [], [], [], [], []
    for class_id in class_ids:
        local = np.flatnonzero(y == class_id)
        if not len(local):
            continue
        class_x = x[local]
        count = min(int(budget), len(local))
        if method == "random":
            rng = np.random.default_rng(seed + int(class_id) * 1009)
            selected_local = rng.choice(len(local), size=count, replace=False)
            assignment = np.argmax(class_x @ class_x[selected_local].T, axis=1)
        elif method == "centroid":
            center = normalize(class_x.mean(axis=0, keepdims=True))
            selected_local = np.array([int(np.argmax(class_x @ center[0]))])
            assignment = np.zeros(len(local), dtype=np.int64)
        else:
            km = MiniBatchKMeans(
                n_clusters=count,
                random_state=seed + int(class_id),
                n_init=1,
                batch_size=min(2048, max(256, len(local))),
                max_iter=75,
                reassignment_ratio=0.01,
            ).fit(class_x)
            centers = normalize(km.cluster_centers_)
            selected_local = []
            used = set()
            for center in centers:
                for candidate in np.argsort(-(class_x @ center)):
                    if int(candidate) not in used:
                        selected_local.append(int(candidate))
                        used.add(int(candidate))
                        break
            selected_local = np.asarray(selected_local)
            assignment = np.argmax(class_x @ class_x[selected_local].T, axis=1)
        selected_global = local[selected_local]
        selected_x = x[selected_global]
        for cluster_id, (global_id, vector) in enumerate(
            zip(selected_global, selected_x, strict=False)
        ):
            members = class_x[assignment == cluster_id]
            compact = float(np.mean(members @ vector)) if len(members) else 0.0
            _, neighbor_ids = all_index.search(vector[None].astype(np.float32), min(33, len(x)))
            neighbor_ids = neighbor_ids[0]
            neighbor_ids = neighbor_ids[neighbor_ids != global_id][:32]
            pure = float(np.mean(y[neighbor_ids] == class_id)) if len(neighbor_ids) else 1.0
            align = float((vector @ text_prototypes[int(class_id)] + 1.0) / 2.0)
            protos.append(vector)
            labels.append(int(class_id))
            sources.append(int(global_id))
            compactness.append(compact)
            purity.append(pure)
            alignment.append(align)
    memory = {
        "prototypes": normalize(np.stack(protos)),
        "labels": np.asarray(labels, np.int64),
        "source_idx": np.asarray(sources, np.int64),
        "compactness": minmax(compactness),
        "purity": minmax(purity),
        "alignment": minmax(alignment),
        "method": method,
        "budget": int(budget),
        "seed": int(seed),
    }
    memory["reliability"] = (
        np.ones(len(labels), np.float32)
        if method != "evidencemem"
        else np.clip(
            0.45 * memory["compactness"] + 0.35 * memory["purity"] + 0.20 * memory["alignment"],
            0.05,
            1.0,
        )
    )
    return memory


def memory_file(method, budget, seed, suffix=""):
    return CACHE_DIR / f"memory_{method}_b{budget}_s{seed}{suffix}.npz"


def fit_or_load_memory(x, y, budget, method, seed, suffix="", text_prototypes=None):
    # Cardinality and encoder identity isolate validation, paper, and model caches.
    encoder_id = hashlib.sha256(
        f"{OPENCLIP_MODEL}|{OPENCLIP_WEIGHTS}|{x.shape[1]}".encode()
    ).hexdigest()[:8]
    path = memory_file(method, budget, seed, f"_n{len(x)}_e{encoder_id}{suffix}")
    if path.exists():
        loaded = np.load(path, allow_pickle=False)
        return {key: loaded[key] for key in loaded.files}
    memory = fit_memory(x, y, budget, method, seed, text_prototypes=text_prototypes)
    with tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix=".npz", delete=False) as handle:
        tmp_name = handle.name
    serializable = {key: value for key, value in memory.items() if isinstance(value, np.ndarray)}
    np.savez_compressed(tmp_name, **serializable)
    os.replace(tmp_name, path)
    return memory


def copy_with_reliability(memory, weights):
    result = dict(memory)
    raw = sum(
        float(weights.get(name, 0.0)) * memory[name]
        for name in ("compactness", "purity", "alignment")
    )
    result["reliability"] = np.clip(raw, 0.05, 1.0).astype(np.float32)
    return result


def search(vectors, queries, k):
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    return index.search(
        np.ascontiguousarray(normalize(queries), dtype=np.float32), min(int(k), len(vectors))
    )


def visual_scores(memory, queries, k, n_classes=10):
    similarities, ids = search(memory["prototypes"], queries, k)
    scores = np.full((len(queries), n_classes), -1.0, dtype=np.float32)
    for row in range(len(queries)):
        for class_id in range(n_classes):
            mask = memory["labels"][ids[row]] == class_id
            if np.any(mask):
                sims = similarities[row, mask]
                rel = memory["reliability"][ids[row, mask]]
                scores[row, class_id] = float(np.sum(sims * rel) / np.clip(np.sum(rel), 1e-8, None))
    return scores


def memory_scores(memory, queries, alpha, k, text_prototypes=None):
    text_prototypes = TEXT_PROTOTYPES if text_prototypes is None else normalize(text_prototypes)
    visual = visual_scores(memory, queries, k, n_classes=len(text_prototypes))
    text = normalize(queries) @ text_prototypes.T
    fused = (1.0 - float(alpha)) * visual + float(alpha) * text
    return fused, visual, text


def full_knn_scores(train_x, train_y, queries, k=20, n_classes=10):
    similarities, ids = search(train_x, queries, k)
    scores = np.full((len(queries), n_classes), -1.0, dtype=np.float32)
    for class_id in range(n_classes):
        mask = train_y[ids] == class_id
        weighted = np.where(mask, np.maximum(similarities, 0.0), 0.0)
        scores[:, class_id] = weighted.sum(1) / np.clip(mask.sum(1), 1, None)
    return scores

In [ ]:
# Metrics, validation-only calibration, confidence intervals, and result persistence.
def softmax_np(scores, temperature):
    logits = np.asarray(scores, np.float64) / float(temperature)
    logits -= logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def expected_calibration_error(probs, labels, bins=15):
    confidence, prediction = probs.max(1), probs.argmax(1)
    edges = np.linspace(0.0, 1.0, bins + 1)
    value = 0.0
    for lo, hi in zip(edges[:-1], edges[1:], strict=False):
        mask = (confidence > lo) & (confidence <= hi)
        if np.any(mask):
            value += mask.mean() * abs(
                (prediction[mask] == labels[mask]).mean() - confidence[mask].mean()
            )
    return float(value)


def select_temperature(val_scores, labels):
    losses = {
        temp: log_loss(labels, softmax_np(val_scores, temp), labels=np.arange(val_scores.shape[1]))
        for temp in CFG.temperatures
    }
    return min(losses, key=losses.get)


def evaluate_scores(name, test_scores, labels, temperature, seed, extra=None):
    probs = softmax_np(test_scores, temperature)
    pred = probs.argmax(1)
    n_classes = test_scores.shape[1]
    one_hot = np.eye(n_classes)[labels]
    row = {
        "method": name,
        "seed": seed,
        "n": len(labels),
        "accuracy": accuracy_score(labels, pred),
        "balanced_accuracy": balanced_accuracy_score(labels, pred),
        "macro_f1": f1_score(labels, pred, average="macro"),
        "nll": log_loss(labels, probs, labels=np.arange(n_classes)),
        "brier": float(np.mean(np.sum((probs - one_hot) ** 2, axis=1))),
        "ece_15": expected_calibration_error(probs, labels),
        "temperature": temperature,
    }
    if extra:
        row.update(extra)
    return row, pred, probs


def paired_bootstrap(correct_a, correct_b, seed=2026, draws=4000):
    correct_a, correct_b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    differences = correct_a - correct_b
    rng = np.random.default_rng(seed)
    samples = np.empty(draws)
    for start in range(0, draws, 200):
        stop = min(start + 200, draws)
        ids = rng.integers(0, len(differences), size=(stop - start, len(differences)))
        samples[start:stop] = differences[ids].mean(1)
    return {
        "delta": float(differences.mean()),
        "ci_low": float(np.quantile(samples, 0.025)),
        "ci_high": float(np.quantile(samples, 0.975)),
        "p_two_sided": float(2 * min((samples <= 0).mean(), (samples >= 0).mean())),
    }


def mcnemar_exact(correct_a, correct_b):
    # Exact two-sided binomial form; no asymptotic assumption for discordant predictions.
    from scipy.stats import binomtest

    n10 = int(np.sum(np.asarray(correct_a) & ~np.asarray(correct_b)))
    n01 = int(np.sum(~np.asarray(correct_a) & np.asarray(correct_b)))
    p = 1.0 if n10 + n01 == 0 else binomtest(min(n10, n01), n10 + n01, 0.5).pvalue
    return {"a_only_correct": n10, "b_only_correct": n01, "p_exact": float(p)}

## 3. Main classification table

Fusion weight, retrieval `k`, and calibration temperature are selected using CIFAR-10
validation labels. The selected setting is then applied once to the official test subset.
Stored prediction files permit paired tests and later error analysis.

In [ ]:
classification_rows = []
prediction_bank = {}
selected_settings = {}
tuning_rows = []

for seed in CFG.seeds:
    seed_everything(seed)
    memories = {
        "Random memory": fit_or_load_memory(X_train, y_train, CFG.default_budget, "random", seed),
        "Centroid": fit_or_load_memory(X_train, y_train, 1, "centroid", seed),
        "KMeans medoids": fit_or_load_memory(X_train, y_train, CFG.default_budget, "medoid", seed),
        "EvidenceMem": fit_or_load_memory(
            X_train, y_train, CFG.default_budget, "evidencemem", seed
        ),
    }

    # Tune EvidenceMem fusion and k on validation only.
    tuning = []
    for k in CFG.topk_grid:
        for alpha in CFG.alpha_grid:
            val_scores, _, _ = memory_scores(memories["EvidenceMem"], X_val, alpha, k)
            val_accuracy = accuracy_score(y_val, val_scores.argmax(1))
            tuning.append((val_accuracy, -k, alpha, k))
            tuning_rows.append(
                {"seed": seed, "alpha": alpha, "k": k, "validation_accuracy": val_accuracy}
            )
    _, _, best_alpha, best_k = max(tuning)
    selected_settings[seed] = {"alpha": best_alpha, "k": best_k}

    score_sets = {}
    val_score_sets = {}
    val_score_sets["CLIP zero-shot"] = X_val @ TEXT_PROTOTYPES.T
    score_sets["CLIP zero-shot"] = X_test @ TEXT_PROTOTYPES.T
    val_score_sets["Full kNN"] = full_knn_scores(X_train, y_train, X_val, k=20)
    score_sets["Full kNN"] = full_knn_scores(X_train, y_train, X_test, k=20)
    for name, memory in memories.items():
        alpha = best_alpha if name == "EvidenceMem" else 0.0
        label = "EvidenceMem fused" if name == "EvidenceMem" else name
        val_score_sets[label] = memory_scores(memory, X_val, alpha, best_k)[0]
        score_sets[label] = memory_scores(memory, X_test, alpha, best_k)[0]
    # Keep the visual endpoint as an explicit fusion ablation.
    val_score_sets["EvidenceMem visual"] = memory_scores(
        memories["EvidenceMem"], X_val, 0.0, best_k
    )[0]
    score_sets["EvidenceMem visual"] = memory_scores(memories["EvidenceMem"], X_test, 0.0, best_k)[
        0
    ]

    # Fixed, conventional linear-probe baseline. Its regularization is not tuned on test.
    probe = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=seed)
    probe.fit(X_train, y_train)
    val_score_sets["Linear probe"] = probe.decision_function(X_val)
    score_sets["Linear probe"] = probe.decision_function(X_test)

    for name, test_scores in score_sets.items():
        temperature = select_temperature(val_score_sets[name], y_val)
        extra = {
            "budget_per_class": (
                CFG.default_budget
                if "memory" in name.lower() or "medoid" in name.lower() or "EvidenceMem" in name
                else np.nan
            ),
            "selected_alpha": best_alpha if name == "EvidenceMem fused" else np.nan,
            "selected_k": best_k if name.startswith("EvidenceMem") else np.nan,
        }
        row, pred, probs = evaluate_scores(name, test_scores, y_test, temperature, seed, extra)
        classification_rows.append(row)
        prediction_bank[(seed, name)] = pred
        np.savez_compressed(
            RUN_DIR / f"predictions_{name.lower().replace(' ', '_')}_s{seed}.npz",
            labels=y_test,
            predictions=pred,
            probabilities=probs,
            scores=test_scores,
        )
    atomic_csv(pd.DataFrame(classification_rows), RUN_DIR / "classification_results.csv")
    atomic_csv(pd.DataFrame(tuning_rows), RUN_DIR / "fusion_topk_validation.csv")

classification_df = pd.DataFrame(classification_rows)
summary_df = (
    classification_df.groupby("method", as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        ece_mean=("ece_15", "mean"),
        nll_mean=("nll", "mean"),
    )
    .fillna(0.0)
    .sort_values("accuracy_mean", ascending=False)
)
atomic_csv(summary_df, RUN_DIR / "classification_summary.csv")
atomic_json(RUN_DIR / "selected_hyperparameters.json", selected_settings)
display(summary_df)

first_seed = CFG.seeds[0]
candidate_names = [
    name
    for name in prediction_bank
    if name[0] == first_seed and not name[1].startswith("EvidenceMem")
]
strongest_key = max(candidate_names, key=lambda key: accuracy_score(y_test, prediction_bank[key]))
ev_correct = prediction_bank[(first_seed, "EvidenceMem fused")] == y_test
base_correct = prediction_bank[strongest_key] == y_test
paired_result = {
    "comparison": f"EvidenceMem fused vs {strongest_key[1]}",
    "bootstrap": paired_bootstrap(ev_correct, base_correct),
    "mcnemar": mcnemar_exact(ev_correct, base_correct),
}
atomic_json(RUN_DIR / "paired_test.json", paired_result)
paired_result

In [ ]:
# Optional supervised ResNet-18 reference. This trains on pixels, so it is reported separately.
def run_resnet18_reference():
    from torchvision.models import ResNet18_Weights, resnet18

    weights = ResNet18_Weights.DEFAULT
    train_transform = torchvision.transforms.Compose(
        [
            torchvision.transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
            torchvision.transforms.RandomHorizontalFlip(),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(weights.transforms().mean, weights.transforms().std),
        ]
    )
    eval_transform = weights.transforms()
    train_ds = Subset(CIFAR10(DATA_DIR, train=True, transform=train_transform), train_idx.tolist())
    val_ds = Subset(CIFAR10(DATA_DIR, train=True, transform=eval_transform), val_idx.tolist())
    test_ds = Subset(CIFAR10(DATA_DIR, train=False, transform=eval_transform), test_idx.tolist())
    loaders = {
        "train": DataLoader(
            train_ds, batch_size=128, shuffle=True, num_workers=CFG.num_workers, pin_memory=True
        ),
        "val": DataLoader(
            val_ds, batch_size=256, shuffle=False, num_workers=CFG.num_workers, pin_memory=True
        ),
        "test": DataLoader(
            test_ds, batch_size=256, shuffle=False, num_workers=CFG.num_workers, pin_memory=True
        ),
    }
    network = resnet18(weights=weights)
    network.fc = torch.nn.Linear(network.fc.in_features, 10)
    network.to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    best_state, best_val = None, -1.0
    for epoch in range(CFG.resnet_epochs):
        network.train()
        for images, labels in tqdm(loaders["train"], desc=f"ResNet epoch {epoch + 1}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = F.cross_entropy(network(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        network.eval()
        val_pred, val_labels = [], []
        with torch.inference_mode():
            for images, labels in loaders["val"]:
                val_pred.extend(network(images.to(DEVICE)).argmax(1).cpu().tolist())
                val_labels.extend(labels.tolist())
        val_accuracy = accuracy_score(val_labels, val_pred)
        if val_accuracy > best_val:
            best_val = val_accuracy
            best_state = {
                key: value.detach().cpu().clone() for key, value in network.state_dict().items()
            }
    network.load_state_dict(best_state)
    network.eval()
    scores, labels = [], []
    with torch.inference_mode():
        for images, batch_labels in loaders["test"]:
            scores.append(network(images.to(DEVICE)).float().cpu().numpy())
            labels.append(batch_labels.numpy())
    scores, labels = np.concatenate(scores), np.concatenate(labels)
    row, pred, probs = evaluate_scores(
        "ResNet-18 supervised", scores, labels, 1.0, CFG.seeds[0], {"epochs": CFG.resnet_epochs}
    )
    atomic_json(RUN_DIR / "resnet18_result.json", row)
    return row


if RUN_RESNET18:
    display(pd.DataFrame([run_resnet18_reference()]))
else:
    print("ResNet-18 reference skipped. Set RUN_RESNET18=True to run it.")

## 3b. Secondary CIFAR-100 classification scaling experiment (paper mode)

This experiment is disabled in the quick validation run to keep its wall time bounded.
Paper mode repeats the frozen-embedding comparison on 100 classes, using a separate
stratified validation split and separate memory caches. CIFAR-100 test labels are not
used in the CIFAR-10 OOD selection above or in CIFAR-100 hyperparameter selection.

In [ ]:
if RUN_CIFAR100_CLASSIFICATION:
    c100_train_cls = CIFAR100(DATA_DIR, train=True, transform=clip_preprocess, download=True)
    c100_train_idx, c100_val_idx = choose_stratified(
        c100_train_cls.targets, 45_000, 5_000, seed=2026
    )
    X100_train, y100_train = encode_dataset(
        "c100_cls_train", Subset(c100_train_cls, c100_train_idx.tolist())
    )
    X100_val, y100_val = encode_dataset(
        "c100_cls_val", Subset(c100_train_cls, c100_val_idx.tolist())
    )
    X100_test, y100_test = X_c100, y_c100  # paper mode encoded the full test set above
    text100 = encode_text_prototypes(
        tuple(name.replace("_", " ") for name in c100_train_cls.classes)
    )
    c100_rows, c100_tuning = [], []
    for seed in CFG.seeds:
        medoids100 = fit_or_load_memory(
            X100_train,
            y100_train,
            CFG.default_budget,
            "medoid",
            seed,
            suffix="_c100",
            text_prototypes=text100,
        )
        evidence100 = fit_or_load_memory(
            X100_train,
            y100_train,
            CFG.default_budget,
            "evidencemem",
            seed,
            suffix="_c100",
            text_prototypes=text100,
        )
        best = None
        for k in CFG.topk_grid:
            for alpha in CFG.alpha_grid:
                scores = memory_scores(evidence100, X100_val, alpha, k, text_prototypes=text100)[0]
                row = {
                    "seed": seed,
                    "alpha": alpha,
                    "k": k,
                    "validation_accuracy": accuracy_score(y100_val, scores.argmax(1)),
                }
                c100_tuning.append(row)
                candidate = (row["validation_accuracy"], -k, alpha, k)
                best = candidate if best is None or candidate > best else best
        _, _, alpha100, k100 = best
        val_sets = {
            "CLIP zero-shot": X100_val @ text100.T,
            "Full kNN": full_knn_scores(X100_train, y100_train, X100_val, k=20, n_classes=100),
            "KMeans medoids": memory_scores(
                medoids100, X100_val, 0.0, k100, text_prototypes=text100
            )[0],
            "EvidenceMem fused": memory_scores(
                evidence100, X100_val, alpha100, k100, text_prototypes=text100
            )[0],
        }
        test_sets = {
            "CLIP zero-shot": X100_test @ text100.T,
            "Full kNN": full_knn_scores(X100_train, y100_train, X100_test, k=20, n_classes=100),
            "KMeans medoids": memory_scores(
                medoids100, X100_test, 0.0, k100, text_prototypes=text100
            )[0],
            "EvidenceMem fused": memory_scores(
                evidence100, X100_test, alpha100, k100, text_prototypes=text100
            )[0],
        }
        probe100 = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=seed)
        probe100.fit(X100_train, y100_train)
        val_sets["Linear probe"] = probe100.decision_function(X100_val)
        test_sets["Linear probe"] = probe100.decision_function(X100_test)
        for name, test_scores in test_sets.items():
            temperature = select_temperature(val_sets[name], y100_val)
            result, pred, probs = evaluate_scores(
                name,
                test_scores,
                y100_test,
                temperature,
                seed,
                {
                    "dataset": "CIFAR-100",
                    "selected_alpha": alpha100 if name == "EvidenceMem fused" else np.nan,
                    "selected_k": k100 if name == "EvidenceMem fused" else np.nan,
                },
            )
            c100_rows.append(result)
            np.savez_compressed(
                RUN_DIR / f"c100_predictions_{name.lower().replace(' ', '_')}_s{seed}.npz",
                labels=y100_test,
                predictions=pred,
                probabilities=probs,
            )
        atomic_csv(pd.DataFrame(c100_rows), RUN_DIR / "cifar100_classification_results.csv")
        atomic_csv(pd.DataFrame(c100_tuning), RUN_DIR / "cifar100_fusion_topk_validation.csv")
    display(
        pd.DataFrame(c100_rows)
        .groupby("method", as_index=False)
        .agg(accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"))
        .fillna(0)
    )
else:
    print(
        "CIFAR-100 classification skipped in validation mode; it runs automatically in paper mode."
    )

## 4. Equal-count memory-budget curves

Random exemplars, plain medoids, and EvidenceMem receive the same number of stored
images per class. Each partial table is saved immediately, making the long paper run
resumable from cached memories.

In [ ]:
budget_path = RUN_DIR / "memory_budget_results.csv"
budget_rows = [] if not budget_path.exists() else pd.read_csv(budget_path).to_dict("records")
finished = {(int(row["seed"]), int(row["budget"]), row["method"]) for row in budget_rows}
for seed in CFG.seeds:
    setting = selected_settings[seed]
    for budget in CFG.budgets:
        for method, label in (
            ("random", "Random"),
            ("medoid", "Plain medoids"),
            ("evidencemem", "EvidenceMem"),
        ):
            if (seed, budget, label) in finished:
                continue
            memory = fit_or_load_memory(X_train, y_train, budget, method, seed)
            alpha = setting["alpha"] if method == "evidencemem" else 0.0
            val_scores = memory_scores(memory, X_val, alpha, setting["k"])[0]
            test_scores = memory_scores(memory, X_test, alpha, setting["k"])[0]
            temp = select_temperature(val_scores, y_val)
            start = time.perf_counter()
            _ = memory_scores(memory, X_test[: min(1000, len(X_test))], alpha, setting["k"])[0]
            latency_ms = 1000 * (time.perf_counter() - start) / min(1000, len(X_test))
            row, _, _ = evaluate_scores(label, test_scores, y_test, temp, seed)
            row.update(
                {
                    "budget": budget,
                    "stored_images": len(memory["labels"]),
                    "latency_ms_per_query": latency_ms,
                    "memory_mib_float32": (
                        memory["prototypes"].nbytes
                        + memory["labels"].nbytes
                        + memory["reliability"].nbytes
                    )
                    / 2**20,
                }
            )
            budget_rows.append(row)
            atomic_csv(pd.DataFrame(budget_rows), budget_path)

budget_df = pd.DataFrame(budget_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.lineplot(
    data=budget_df, x="budget", y="accuracy", hue="method", marker="o", errorbar="sd", ax=axes[0]
)
sns.lineplot(
    data=budget_df,
    x="stored_images",
    y="latency_ms_per_query",
    hue="method",
    marker="o",
    errorbar="sd",
    ax=axes[1],
    legend=False,
)
axes[0].set_xscale("log")
axes[0].set_title("Accuracy at equal memory count")
axes[1].set_xscale("log")
axes[1].set_title("Warm retrieval latency")
fig.tight_layout()
fig.savefig(RUN_DIR / "memory_budget_curve.pdf", bbox_inches="tight")
plt.show()

## 5. Reliability and fusion ablations

These rows isolate compactness, purity, text alignment, reliability weighting, and
visual-text fusion while keeping medoids and the test set fixed.

In [ ]:
ablation_rows = []
for seed in CFG.seeds:
    setting = selected_settings[seed]
    base = fit_or_load_memory(X_train, y_train, CFG.default_budget, "evidencemem", seed)
    variants = {
        "No reliability": ({**base, "reliability": np.ones(len(base["labels"]), np.float32)}, 0.0),
        "Compactness only": (copy_with_reliability(base, {"compactness": 1.0}), 0.0),
        "Purity only": (copy_with_reliability(base, {"purity": 1.0}), 0.0),
        "Alignment only": (copy_with_reliability(base, {"alignment": 1.0}), 0.0),
        "Full reliability visual": (base, 0.0),
        "Full EvidenceMem fused": (base, setting["alpha"]),
    }
    for name, (memory, alpha) in variants.items():
        val_scores = memory_scores(memory, X_val, alpha, setting["k"])[0]
        test_scores = memory_scores(memory, X_test, alpha, setting["k"])[0]
        temp = select_temperature(val_scores, y_val)
        row, _, _ = evaluate_scores(name, test_scores, y_test, temp, seed)
        row.update({"alpha": alpha, "k": setting["k"]})
        ablation_rows.append(row)
atomic_csv(pd.DataFrame(ablation_rows), RUN_DIR / "ablation_results.csv")
display(
    pd.DataFrame(ablation_rows)
    .groupby("method", as_index=False)["accuracy"]
    .agg(["mean", "std"])
    .fillna(0)
)

## 6. Near- and far-OOD rejection

Higher confidence means “more likely CIFAR-10.” FPR95 uses a threshold chosen solely
from held-out CIFAR-10 validation confidence (95% ID acceptance), then measures the
fraction of OOD examples incorrectly accepted. AUROC/AUPR use the untouched ID/OOD
test examples. The disagreement score is fixed in advance; it is not tuned on OOD.

In [ ]:
def js_divergence(p, q):
    m = 0.5 * (p + q)
    return 0.5 * np.sum(p * np.log(np.clip(p / m, 1e-12, None)), axis=1) + 0.5 * np.sum(
        q * np.log(np.clip(q / m, 1e-12, None)), axis=1
    )


def ood_metrics(id_val_conf, id_test_conf, ood_conf):
    threshold = float(np.quantile(id_val_conf, 0.05))
    labels = np.r_[np.zeros(len(id_test_conf), dtype=int), np.ones(len(ood_conf), dtype=int)]
    ood_score = -np.r_[id_test_conf, ood_conf]
    return {
        "auroc": roc_auc_score(labels, ood_score),
        "aupr_ood": average_precision_score(labels, ood_score),
        "fpr95": float(np.mean(ood_conf >= threshold)),
        "id_threshold": threshold,
    }


ood_rows = []
for seed in CFG.seeds:
    setting = selected_settings[seed]
    memory = fit_or_load_memory(X_train, y_train, CFG.default_budget, "evidencemem", seed)
    datasets = {"CIFAR-100 (near)": X_c100, "SVHN (far)": X_svhn}
    # Validation and ID-test confidence for each fixed score.
    val_fused, val_visual, val_text = memory_scores(memory, X_val, setting["alpha"], setting["k"])
    id_fused, id_visual, id_text = memory_scores(memory, X_test, setting["alpha"], setting["k"])
    temp_fused = select_temperature(val_fused, y_val)
    temp_visual = select_temperature(val_visual, y_val)
    temp_text = select_temperature(val_text, y_val)
    val_text_probs, val_visual_probs, val_fused_probs = (
        softmax_np(val_text, temp_text),
        softmax_np(val_visual, temp_visual),
        softmax_np(val_fused, temp_fused),
    )
    id_text_probs, id_visual_probs, id_fused_probs = (
        softmax_np(id_text, temp_text),
        softmax_np(id_visual, temp_visual),
        softmax_np(id_fused, temp_fused),
    )

    def confidence_set(text_probs, visual_probs, fused_probs, queries, prototype_vectors):
        sorted_fused = np.sort(fused_probs, axis=1)
        entropy = -np.sum(fused_probs * np.log(np.clip(fused_probs, 1e-12, None)), axis=1)
        max_similarity = search(prototype_vectors, queries, 1)[0][:, 0]
        return {
            "CLIP text MSP": text_probs.max(1),
            "Maximum prototype similarity": max_similarity,
            "EvidenceMem MSP": fused_probs.max(1),
            "Predictive entropy": 1.0 - entropy / math.log(10),
            "Probability margin": sorted_fused[:, -1] - sorted_fused[:, -2],
            "Disagreement-aware": fused_probs.max(1)
            * (1.0 - js_divergence(text_probs, visual_probs) / math.log(2)),
        }

    val_conf = confidence_set(
        val_text_probs, val_visual_probs, val_fused_probs, X_val, memory["prototypes"]
    )
    id_conf = confidence_set(
        id_text_probs, id_visual_probs, id_fused_probs, X_test, memory["prototypes"]
    )
    for dataset_name, x_ood in datasets.items():
        ood_fused, ood_visual, ood_text = memory_scores(
            memory, x_ood, setting["alpha"], setting["k"]
        )
        ood_text_probs = softmax_np(ood_text, temp_text)
        ood_visual_probs = softmax_np(ood_visual, temp_visual)
        ood_fused_probs = softmax_np(ood_fused, temp_fused)
        ood_conf = confidence_set(
            ood_text_probs, ood_visual_probs, ood_fused_probs, x_ood, memory["prototypes"]
        )
        for method, confidence in ood_conf.items():
            row = ood_metrics(val_conf[method], id_conf[method], confidence)
            row.update({"seed": seed, "dataset": dataset_name, "method": method})
            ood_rows.append(row)
atomic_csv(pd.DataFrame(ood_rows), RUN_DIR / "ood_results.csv")
display(
    pd.DataFrame(ood_rows)
    .groupby(["dataset", "method"], as_index=False)[["auroc", "aupr_ood", "fpr95"]]
    .mean()
)

## 7. Class insertion without encoder updates

Classes 0–5 form the initial memory. Classes 6–9 are inserted from a few labeled
examples. Old prototypes and the encoder are never changed. Reported forgetting can
still be nonzero because new prototypes may compete with old ones at retrieval time.

In [ ]:
def insert_memory(base, x_support, y_support, budget, seed):
    addition = fit_memory(x_support, y_support, budget, "evidencemem", seed)
    result = {}
    for key in (
        "prototypes",
        "labels",
        "source_idx",
        "compactness",
        "purity",
        "alignment",
        "reliability",
    ):
        result[key] = np.concatenate([base[key], addition[key]])
    return result


continual_rows = []
old_classes, new_classes = np.arange(6), np.arange(6, 10)
for seed in CFG.seeds:
    seed_everything(seed)
    old_mask = np.isin(y_train, old_classes)
    old_memory = fit_memory(
        X_train[old_mask], y_train[old_mask], CFG.default_budget, "evidencemem", seed
    )
    old_test_mask = np.isin(y_test, old_classes)
    # Visual-only keeps the protocol valid before all text-labelled classes have examples.
    before_scores = visual_scores(old_memory, X_test[old_test_mask], selected_settings[seed]["k"])
    before_old_acc = accuracy_score(y_test[old_test_mask], before_scores.argmax(1))
    for shots in CFG.insertion_shots:
        support_ids = []
        rng = np.random.default_rng(seed + shots)
        for class_id in new_classes:
            candidates = np.flatnonzero(y_train == class_id)
            support_ids.extend(
                rng.choice(candidates, size=min(shots, len(candidates)), replace=False).tolist()
            )
        support_ids = np.asarray(support_ids)
        start = time.perf_counter()
        updated = insert_memory(
            old_memory,
            X_train[support_ids],
            y_train[support_ids],
            min(shots, CFG.default_budget),
            seed,
        )
        insertion_seconds = time.perf_counter() - start
        scores = visual_scores(updated, X_test, selected_settings[seed]["k"])
        pred = scores.argmax(1)
        old_acc = accuracy_score(y_test[old_test_mask], pred[old_test_mask])
        new_mask = np.isin(y_test, new_classes)
        continual_rows.append(
            {
                "seed": seed,
                "shots_per_new_class": shots,
                "overall_accuracy": accuracy_score(y_test, pred),
                "old_accuracy_before": before_old_acc,
                "old_accuracy_after": old_acc,
                "new_class_accuracy": accuracy_score(y_test[new_mask], pred[new_mask]),
                "forgetting": before_old_acc - old_acc,
                "insertion_seconds": insertion_seconds,
                "encoder_updates": 0,
                "old_prototypes_changed": 0,
            }
        )
atomic_csv(pd.DataFrame(continual_rows), RUN_DIR / "continual_insertion_results.csv")
display(pd.DataFrame(continual_rows))

## 8. Evidence quality and qualitative failures

Evidence Precision@k is the fraction of retrieved stored images whose class label
matches the query label. This is **decision evidence**, not a causal explanation.
The montage deliberately includes incorrect predictions when available.

In [ ]:
seed = CFG.seeds[0]
setting = selected_settings[seed]
evidence_memory = fit_or_load_memory(X_train, y_train, CFG.default_budget, "evidencemem", seed)
similarities, retrieved = search(evidence_memory["prototypes"], X_test, 5)
retrieved_labels = evidence_memory["labels"][retrieved]
evidence_rows = []
for method, memory in {
    "Random": fit_or_load_memory(X_train, y_train, CFG.default_budget, "random", seed),
    "Plain medoids": fit_or_load_memory(X_train, y_train, CFG.default_budget, "medoid", seed),
    "EvidenceMem": evidence_memory,
}.items():
    _, method_retrieved = search(memory["prototypes"], X_test, 5)
    method_labels = memory["labels"][method_retrieved]
    for k in (1, 3, 5):
        per_query = np.mean(method_labels[:, :k] == y_test[:, None], axis=1)
        evidence_rows.append(
            {
                "method": method,
                "k": k,
                "precision_at_k": float(per_query.mean()),
                "queries": len(y_test),
                "memory_images": len(memory["labels"]),
            }
        )
atomic_csv(pd.DataFrame(evidence_rows), RUN_DIR / "evidence_precision.csv")
display(pd.DataFrame(evidence_rows))

test_scores = memory_scores(evidence_memory, X_test, setting["alpha"], setting["k"])[0]
test_pred = test_scores.argmax(1)
wrong = np.flatnonzero(test_pred != y_test)
right = np.flatnonzero(test_pred == y_test)
rng = np.random.default_rng(2026)
chosen = np.r_[
    rng.choice(wrong, min(3, len(wrong)), replace=False),
    rng.choice(right, min(3, len(right)), replace=False),
]
fig, axes = plt.subplots(len(chosen), 4, figsize=(10, 2.5 * len(chosen)), squeeze=False)
for row, query_local in enumerate(chosen):
    query_original = int(test_idx[query_local])
    query_image, _ = c10_test_raw[query_original]
    axes[row, 0].imshow(query_image)
    axes[row, 0].set_title(
        f"Query: {CLASS_NAMES[y_test[query_local]]}\nPred: {CLASS_NAMES[test_pred[query_local]]}"
    )
    for col in range(1, 4):
        proto_id = int(retrieved[query_local, col - 1])
        train_local = int(evidence_memory["source_idx"][proto_id])
        train_original = int(train_idx[train_local])
        image, _ = c10_train_raw[train_original]
        axes[row, col].imshow(image)
        evidence_label = CLASS_NAMES[evidence_memory["labels"][proto_id]]
        similarity = similarities[query_local, col - 1]
        axes[row, col].set_title(f"Evidence {col}: {evidence_label}\nsim={similarity:.3f}")
    for axis in axes[row]:
        axis.axis("off")
fig.tight_layout()
fig.savefig(RUN_DIR / "qualitative_evidence.pdf", bbox_inches="tight")
plt.show()

## 9. Embedding-noise robustness

Noise is applied only to the stored training embeddings, followed by renormalization.
The frozen encoder, validation queries, and test queries remain unchanged. This tests
memory sensitivity rather than image-corruption robustness.

In [ ]:
noise_rows = []
if RUN_NOISE_ROBUSTNESS:
    levels = (0.0, 0.05, 0.10) if CFG.mode == "validation" else (0.0, 0.02, 0.05, 0.10, 0.20)
    for seed in CFG.seeds:
        rng = np.random.default_rng(seed)
        setting = selected_settings[seed]
        for sigma in levels:
            noisy_train = (
                normalize(X_train + rng.normal(0, sigma, X_train.shape).astype(np.float32))
                if sigma
                else X_train
            )
            memory = fit_memory(noisy_train, y_train, CFG.default_budget, "evidencemem", seed)
            val_scores = memory_scores(memory, X_val, setting["alpha"], setting["k"])[0]
            test_scores = memory_scores(memory, X_test, setting["alpha"], setting["k"])[0]
            temp = select_temperature(val_scores, y_val)
            row, _, _ = evaluate_scores("EvidenceMem", test_scores, y_test, temp, seed)
            row["embedding_noise_sigma"] = sigma
            noise_rows.append(row)
            atomic_csv(pd.DataFrame(noise_rows), RUN_DIR / "noise_robustness.csv")
    display(pd.DataFrame(noise_rows))
else:
    print("Noise robustness skipped.")

## 10. Claim-level validation and export

The go/no-go gate is intentionally mechanical. A negative result is valid: it means
the contribution should be narrowed or reported as an analysis instead of promoted as
a winning method. `validation` mode verifies the pipeline, not publication readiness.

In [ ]:
def mean_for(frame, column, **filters):
    selected = frame.copy()
    for key, value in filters.items():
        selected = selected[selected[key] == value]
    return float(selected[column].mean()) if len(selected) else float("nan")


classification_df = pd.read_csv(RUN_DIR / "classification_results.csv")
budget_df = pd.read_csv(RUN_DIR / "memory_budget_results.csv")
ood_df = pd.read_csv(RUN_DIR / "ood_results.csv")
evidence_df = pd.read_csv(RUN_DIR / "evidence_precision.csv")

budget_wins = []
for budget in CFG.budgets:
    ev = mean_for(budget_df, "accuracy", method="EvidenceMem", budget=budget)
    plain = mean_for(budget_df, "accuracy", method="Plain medoids", budget=budget)
    random_score = mean_for(budget_df, "accuracy", method="Random", budget=budget)
    budget_wins.append(ev > max(plain, random_score))

fused_acc = mean_for(classification_df, "accuracy", method="EvidenceMem fused")
visual_acc = mean_for(classification_df, "accuracy", method="EvidenceMem visual")
text_acc = mean_for(classification_df, "accuracy", method="CLIP zero-shot")
ood_ev = ood_df[ood_df.method == "Disagreement-aware"].auroc.mean()
ood_base = (
    ood_df[
        ood_df.method.isin(
            [
                "CLIP text MSP",
                "Maximum prototype similarity",
                "Predictive entropy",
                "Probability margin",
            ]
        )
    ]
    .groupby("method")
    .auroc.mean()
    .max()
)
precision_3 = float(
    evidence_df.loc[
        (evidence_df.method == "EvidenceMem") & (evidence_df.k == 3), "precision_at_k"
    ].iloc[0]
)

gate = {
    "mode": CFG.mode,
    "paper_ready_run": CFG.mode == "paper" and len(CFG.seeds) >= 3 and CFG.test_size == 10_000,
    "signals": {
        "reliability_beats_equal_budget_baselines_on_any_budget": bool(any(budget_wins)),
        "fusion_beats_both_endpoints": bool(fused_acc > max(visual_acc, text_acc)),
        "combined_confidence_beats_single_scores_mean_auroc": bool(ood_ev > ood_base),
        "evidence_precision_at_3_at_least_0.80": bool(precision_3 >= 0.80),
    },
}
gate["signals_passed"] = sum(gate["signals"].values())
gate["go_no_go"] = (
    "GO: at least two pilot signals"
    if gate["signals_passed"] >= 2
    else "NO-GO: narrow or reframe the contribution"
)
if CFG.mode != "paper":
    gate["warning"] = (
        "Validation mode checks code and protocol only. Do not cite its "
        "subset/one-seed numbers as final results."
    )
atomic_json(RUN_DIR / "claim_validation.json", gate)
journal("run_completed", gate=gate)

print(json.dumps(gate, indent=2))
print("\nResult files:")
for path in sorted(RUN_DIR.iterdir()):
    print(" -", path.name)

# Make a single downloadable archive. In Colab, uncomment the final two lines to download.
archive = __import__("shutil").make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Archive:", archive)
# from google.colab import files
# files.download(archive)